# Robustness checks for the H1-H4 primary tests

Each primary test is refit under alternative choices so that the effect of the
two data corrections, and the dependence of each result on specific
subgroups, is visible in one place.

1. Weights: the original pooled weights (`WTSAF2YR` or `WTMEC2YR` divided by the
   number of cycles) against the thyroid-subsample-corrected weight
   (`wt_thyroid_2yr`, see `pipeline/nhanes_helpers.py`).
2. The single record flagged `ft3_implausible`, retained or dropped.
3. Euthyroid participants only (TSH 0.4-4.5 mIU/L).
4. Thyroid peroxidase antibody negative participants only (`LBXTPO` < 9 IU/mL).
5. The 2007-2008 cycle alone, where the thyroid panel covered the full exam
   sample and no subsample adjustment is needed.

In [1]:
import warnings

import numpy as np
import pandas as pd
import polars as pl
import svy

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
sex = svy.Cat('RIAGENDR', ref=1)

In [2]:
def load(name, base_weight, n_cycles):
    df = pd.read_csv(f'../data/processed/{name}_analytic_sample.csv', low_memory=False)
    df['w_original'] = df[base_weight] / n_cycles
    df['w_corrected'] = df['wt_thyroid_2yr'] / n_cycles
    df['w_2007_only'] = np.where(df['cycle'] == '2007-2008', df[base_weight], 0)
    df['ratio'] = df['LBXT3F'] / df['LBXT4F']
    df['lipid'] = df['on_lipid_lowering'].astype(int) if 'on_lipid_lowering' in df else 0
    if 'LBXCRP' in df:
        df['log_crp'] = np.log(df['LBXCRP'])
    return df


def zscore(df):
    df = df.copy()
    df['ft3_z'] = (df['LBXT3F'] - df['LBXT3F'].mean()) / df['LBXT3F'].std()
    df['tsh_z'] = (df['LBXTSH1'] - df['LBXTSH1'].mean()) / df['LBXTSH1'].std()
    return df


def fit(df, y, x, wcol, family):
    names = [y] + [v if isinstance(v, str) else v.name for v in x]
    keep = df.dropna(subset=names)
    keep = keep[keep[wcol] > 0]
    cols = list(dict.fromkeys(names + ['SDMVSTRA', 'SDMVPSU', wcol]))
    sample = svy.Sample(pl.from_pandas(keep[cols]), design=svy.Design(stratum='SDMVSTRA', psu='SDMVPSU', wgt=wcol))
    model = svy.GLM(sample)
    model.fit(y=y, x=x, family=family)
    row = model.to_polars().filter(pl.col('term') == x[0]).row(0, named=True)
    return dict(n=len(keep), coef=row['estimate'], ci_low=row['conf_low'], ci_high=row['conf_high'], p=row['p_value'])

In [3]:
h1 = load('h1', 'WTSAF2YR', 3)
h2 = load('h2', 'WTSAF2YR', 3)
h3 = load('h3', 'WTMEC2YR', 2)
h4 = load('h4', 'WTMEC2YR', 3)

tests = {
    'H1 FT3/FT4 ratio -> triglycerides (age+sex)': (h1, 'LBXTR', ['ratio', 'RIDAGEYR', sex], 'gaussian'),
    'H2 free T3 (z) -> metabolic syndrome': (h2, 'metabolic_syndrome', ['ft3_z', 'RIDAGEYR', sex, 'BMXBMI'], 'binomial'),
    'H2 TSH (z) -> metabolic syndrome': (h2, 'metabolic_syndrome', ['tsh_z', 'RIDAGEYR', sex, 'BMXBMI'], 'binomial'),
    'H3 free T3 -> log CRP': (h3, 'log_crp', ['LBXT3F', 'RIDAGEYR', sex, 'BMXBMI'], 'gaussian'),
    'H4 free T3 -> cholesterol (primary)': (h4[~h4['on_lipid_lowering']], 'LBXTC', ['LBXT3F', 'RIDAGEYR', sex, 'BMXBMI'], 'gaussian'),
    'H4 free T3 -> cholesterol (sensitivity)': (h4, 'LBXTC', ['LBXT3F', 'RIDAGEYR', sex, 'BMXBMI', 'lipid'], 'gaussian'),
}

clean = lambda df: df[~df['ft3_implausible']]
variants = {
    'original weights, flagged record kept': (lambda df: df, 'w_original'),
    'original weights, flagged record dropped': (clean, 'w_original'),
    'corrected weights, flagged record kept': (lambda df: df, 'w_corrected'),
    'corrected weights, flagged record dropped (reported)': (clean, 'w_corrected'),
    'reported + euthyroid only (TSH 0.4-4.5)': (lambda df: clean(df)[clean(df)['LBXTSH1'].between(0.4, 4.5)], 'w_corrected'),
    'reported + TPO antibody negative': (lambda df: clean(df)[clean(df)['LBXTPO'] < 9], 'w_corrected'),
    'reported, 2007-2008 cycle only': (clean, 'w_2007_only'),
}

rows = []
for test, (df, y, x, family) in tests.items():
    for variant, (subset, wcol) in variants.items():
        # z-scores are recomputed within each subset so the scale matches the data fitted
        rows.append(dict(test=test, variant=variant, **fit(zscore(subset(df)), y, x, wcol, family)))

results = pd.DataFrame(rows)
pd.set_option('display.width', 200, 'display.max_colwidth', 60)
results.round({'coef': 4, 'ci_low': 4, 'ci_high': 4}).assign(p=results['p'].map('{:.3g}'.format))

,test,variant,n,coef,ci_low,ci_high,p
0,H1 FT3/FT4 ratio -> triglycerides (age+sex),"original weights, flagged record kept",3459,14.8728,7.8230,21.9226,0.000104
1,H1 FT3/FT4 ratio -> triglycerides (age+sex),"original weights, flagged record dropped",3458,18.7119,12.5996,24.8242,1.65e-07
2,H1 FT3/FT4 ratio -> triglycerides (age+sex),"corrected weights, flagged record kept",3459,15.0903,7.1999,22.9808,0.000364
3,H1 FT3/FT4 ratio -> triglycerides (age+sex),"corrected weights, flagged record dropped (reported)",3458,20.3342,13.1900,27.4785,7.36e-07
4,H1 FT3/FT4 ratio -> triglycerides (age+sex),reported + euthyroid only (TSH 0.4-4.5),3259,19.8891,12.3385,27.4396,3.17e-06
5,H1 FT3/FT4 ratio -> triglycerides (age+sex),reported + TPO antibody negative,3118,23.0968,15.1751,31.0185,4.55e-07
6,H1 FT3/FT4 ratio -> triglycerides (age+sex),"reported, 2007-2008 cycle only",2034,15.7726,7.2008,24.3444,0.00158
7,H2 free T3 (z) -> metabolic syndrome,"original weights, flagged record kept",3372,0.3215,-0.0254,0.6684,0.0685
8,H2 free T3 (z) -> metabolic syndrome,"original weights, flagged record dropped",3371,0.3376,0.1506,0.5247,0.000709
9,H2 free T3 (z) -> metabolic syndrome,"corrected weights, flagged record kept",3372,0.3409,-0.0461,0.7278,0.0828


## H2 verdict with the flagged record retained

H2 is decided by AUC rather than by a p-value, so this checks whether free T3
still has the higher AUC when the flagged record is kept.

In [4]:
from sklearn.metrics import roc_auc_score


def weighted_auc(df, predictors):
    df = zscore(df)
    data = pl.from_pandas(df)
    sample = svy.Sample(data, design=svy.Design(stratum='SDMVSTRA', psu='SDMVPSU', wgt='w_corrected'))
    model = svy.GLM(sample)
    model.fit(y='metabolic_syndrome', x=[*predictors, 'RIDAGEYR', sex, 'BMXBMI'], family='binomial')
    pred = model.predict(data)
    valid = ~pd.isna(pred.yhat)
    return roc_auc_score(df.loc[valid, 'metabolic_syndrome'], pred.yhat[valid], sample_weight=df.loc[valid, 'w_corrected'])


for label, df in [('flagged record kept', h2), ('flagged record dropped (reported)', clean(h2))]:
    print(f"{label:<36} AUC free T3 = {weighted_auc(df, ['ft3_z']):.4f}  TSH = {weighted_auc(df, ['tsh_z']):.4f}  covariates only = {weighted_auc(df, []):.4f}")

flagged record kept                  AUC free T3 = 0.8278  TSH = 0.8241  covariates only = 0.8233


flagged record dropped (reported)    AUC free T3 = 0.8289  TSH = 0.8241  covariates only = 0.8232
